# Homework 2 of LLM Zoomcamp
This notebook shows my work towards completing [Homework 2](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2025/02-vector-search/homework.md) of the 2025 cohort of the course LLM Zoomcamp.

Commands to start a Qdrant Docker container:

```bash
docker pull qdrant/qdrant
docker run -p 6333:6333 -p 6334:6334 \
   -v "$(pwd)/data/qdrant:/qdrant/storage:z" \
   qdrant/qdrant
```

In [ ]:
import requests

import numpy as np
import pandas as pd
from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models

In [ ]:
client = QdrantClient("http://localhost:6333")

## Question 1
The minimum value in the embedding array is *-0.11*.

In [ ]:
query = "I just discovered the course. Can I join now?"
embedding_model = "jinaai/jina-embeddings-v2-small-en"  # Creates output vector with dim=512

In [ ]:
embedding = TextEmbedding(model_name=embedding_model)
query_embeddings = list(embedding.embed([query]))[0]

In [ ]:
query_embeddings.shape, min(query_embeddings)

## Question 2
The cosine similarity is *0.9*.

In [ ]:
doc = "Can I still join the course after the start date?"

In [ ]:
np.linalg.norm(query_embeddings)

In [ ]:
doc_embeddings = list(embedding.embed(doc))[0]
query_embeddings.dot(doc_embeddings)

## Question 3
The document index with the highest similarity is *1*.

In [ ]:
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
    'section': 'General course-related questions',
    'question': 'Course - Can I still join the course after the start date?',
    'course': 'data-engineering-zoomcamp'},
    {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
    'section': 'General course-related questions',
    'question': 'Course - Can I follow the course after it finishes?',
    'course': 'data-engineering-zoomcamp'},
    {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
    'section': 'General course-related questions',
    'question': 'Course - When will the course start?',
    'course': 'data-engineering-zoomcamp'},
    {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
    'section': 'General course-related questions',
    'question': 'Course - What can I do before the course starts?',
    'course': 'data-engineering-zoomcamp'},
    {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
    'section': 'General course-related questions',
    'question': 'How can we contribute to the course?',
    'course': 'data-engineering-zoomcamp'}]

In [ ]:
documents_embeddings = []
for doc in documents:
    documents_embeddings.append(list(embedding.embed(doc["text"])))

document_embeddings_np = np.array(documents_embeddings)

In [ ]:
document_embeddings_np.dot(query_embeddings)

## Question 4
The document index with the highest similarity is *0*.

In [ ]:
qa_documents_embeddings = []
for doc in documents:
    qa_documents_embeddings.append(list(embedding.embed(doc["question"] + " " + doc["text"])))

qa_document_embeddings_np = np.array(qa_documents_embeddings)

In [ ]:
qa_document_embeddings_np.dot(query_embeddings)

## Question 5: Smallest model
The smallest text embedding model has a dimension of `384`.

In [ ]:
df_supported_models = pd.DataFrame(TextEmbedding.list_supported_models())
df_supported_models.sort_values("dim")

## Question 6
The highest score in the results in *0.87*.

In [ ]:
docs_url = "https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json"
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

ml_zoomcamp_documents = []

for course in documents_raw:
    course_name = course["course"]
    if course_name != "machine-learning-zoomcamp":
        continue

    for doc in course["documents"]:
        doc["course"] = course_name
        ml_zoomcamp_documents.append(doc)

In [ ]:
client.create_collection(
    collection_name="homework_2_q6",
    vectors_config=models.VectorParams(
        size=384,
        distance=models.Distance.COSINE
    )
)

In [ ]:
embedding_model_q6 = "BAAI/bge-small-en"
ml_zoomcamp_embedding = TextEmbedding(model_name=embedding_model_q6)

In [ ]:
points = []
for idx, doc in enumerate(ml_zoomcamp_documents):
    ml_zoomcamp_document_embedding = list(ml_zoomcamp_embedding.embed(doc["question"] + " " + doc["text"]))[0]
    point = models.PointStruct(
        id=idx, 
        vector=ml_zoomcamp_document_embedding,
        payload={
            "text": doc["question"] + " " + doc["text"],
            "section": doc["section"],
            "course": doc["course"]
        }
    )
    points.append(point)

client.upsert(
    collection_name="homework_2_q6",
    points=points,
    wait=True
)

In [ ]:
def search(query: str, collection_name: str, limit: int = 1):
    results = client.query_points(
        collection_name=collection_name,
        query=models.Document(
            text=query,
            model=embedding_model_q6
        ),
        limit=limit,
        with_payload=True
    )

    return list(results)

In [ ]:
search(query, "homework_2_q6", 1)[0][1][0]